# 🏁 F1 Qualifying Classification - Complete Analysis

**Project:** Formula 1 Qualifying Outcome Prediction  
**Author:** Tomasz Solis  
**Date:** November 2024  

## Objectives

After discovering that regression (predicting exact positions) couldn't beat baseline, we pivot to classification:

1. **Top 3 Finish (Binary):** Will they podium in quali?
2. **Q3 Qualification (Binary):** Will they make top 10?
3. **Qualifying Round (Multi-class):** Which round will they reach? (Q1/Q2/Q3)

## Success Criteria

- Top 3 accuracy > 60% (vs 15% baseline)
- Q3 accuracy > 75% (vs 50% baseline)
- Q2 multi-class > 65% (vs 33% baseline)

In [1]:
# Imports
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.impute import SimpleImputer

import joblib
import json
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

print("✅ Imports complete")

✅ Imports complete


## 1. Load and Explore Data

In [2]:
# Load data
print("📊 Loading F1 data...")
df = pd.read_parquet('../data/features/ml_features_2022_2025.parquet')

print(f"✅ Loaded: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
print(df.columns.tolist())

📊 Loading F1 data...
✅ Loaded: (1888, 55)

Columns (55):
['year', 'event', 'driver', 'session_date', 'team', 'qualifying_position', 'max_throttle_ratio', 'brake_max_g', 'brake_avg_g', 'avg_rainfall', 'avg_track_temp', 'avg_air_temp', 'race_position', 'circuit_avg_position', 'circuit_best_position', 'circuit_worst_position', 'circuit_sessions', 'circuit_std_position', 'circuit_avg_race', 'circuit_best_race', 'recent_avg_position', 'recent_best_position', 'recent_worst_position', 'form_trend', 'races_in_window', 'wet_avg_position', 'dry_avg_position', 'wet_sessions', 'dry_sessions', 'wet_dry_delta', 'team_circuit_avg_position', 'team_circuit_best_position', 'team_circuit_sessions', 'team_momentum', 'team_recent_avg', 'circuit_avg_position_change', 'circuit_std_position_change', 'circuit_abs_position_change', 'circuit_max_gain', 'circuit_max_loss', 'circuit_overtaking_samples', 'driver_avg_position_change', 'driver_std_position_change', 'driver_overtaking_success_rate', 'driver_defensive_

In [3]:
# Filter to qualifying sessions only
df = df[df['qualifying_position'].notna()].copy()
print(f"✅ Rows with qualifying positions: {len(df)}")

# Basic statistics
print(f"\n📊 Data Summary:")
print(f"   Years: {df['year'].min()} - {df['year'].max()}")
print(f"   Events: {df['event'].nunique()}")
print(f"   Drivers: {df['driver'].nunique()}")
print(f"   Teams: {df['team'].nunique() if 'team' in df.columns else 'N/A'}")

✅ Rows with qualifying positions: 1778

📊 Data Summary:
   Years: 2022 - 2025
   Events: 25
   Drivers: 31
   Teams: 13


In [4]:
# Detect sprint weekends
if 'has_sprint_quali_data' in df.columns:
    df['is_sprint_weekend'] = df['has_sprint_quali_data'].fillna(False)
elif 'sprint_quali_throttle' in df.columns:
    df['is_sprint_weekend'] = df['sprint_quali_throttle'].notna()
else:
    # Try to infer from session patterns
    if 'sessions_available' in df.columns:
        df['is_sprint_weekend'] = df['sessions_available'] < 3
    else:
        df['is_sprint_weekend'] = False

sprint_count = df['is_sprint_weekend'].sum()
print(f"🏃 Sprint weekends detected: {sprint_count} ({100*sprint_count/len(df):.1f}%)")

🏃 Sprint weekends detected: 0 (0.0%)


## 2. Create Classification Targets

In [5]:
print("🎯 Creating classification targets...\n")

# Target 1: Q3 Qualification (Top 10)
df['made_q3'] = (df['qualifying_position'] <= 10).astype(int)

# Target 2: Top 3 (Podium positions)
df['top3_quali'] = (df['qualifying_position'] <= 3).astype(int)

# Target 3: Qualifying Round (Multi-class)
def classify_quali_round(position):
    if position <= 10:
        return 'Q3'
    elif position <= 15:
        return 'Q2'
    else:
        return 'Q1'

df['quali_round'] = df['qualifying_position'].apply(classify_quali_round)
quali_round_map = {'Q1': 0, 'Q2': 1, 'Q3': 2}
df['quali_round_encoded'] = df['quali_round'].map(quali_round_map)

# Display distribution
print("✅ Target distribution:")
print(f"   Q3 qualification: {df['made_q3'].mean():.1%} positive")
print(f"   Top 3 in quali:   {df['top3_quali'].mean():.1%} positive")
print(f"\n   Qualifying rounds:")
print(f"      Q1 (P16-20): {(df['quali_round'] == 'Q1').mean():.1%}")
print(f"      Q2 (P11-15): {(df['quali_round'] == 'Q2').mean():.1%}")
print(f"      Q3 (P1-10):  {(df['quali_round'] == 'Q3').mean():.1%}")

🎯 Creating classification targets...

✅ Target distribution:
   Q3 qualification: 50.1% positive
   Top 3 in quali:   15.0% positive

   Qualifying rounds:
      Q1 (P16-20): 24.6%
      Q2 (P11-15): 25.3%
      Q3 (P1-10):  50.1%


## 3. Feature Selection & Data Preparation

In [6]:
print("🔍 Selecting features...\n")

# Exclude non-predictive columns
exclude_cols = [
    'year', 'event', 'driver', 'team', 'session_date',
    'qualifying_position', 'race_position',
    'made_q3', 'top3_quali', 'quali_round', 'quali_round_encoded',
    'beat_teammate', 'has_sprint_quali_data', 'sessions_available'
]

# Get numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
feature_cols = [col for col in numeric_cols if col not in exclude_cols]

print(f"✅ Available features: {len(feature_cols)}")

# Check missing data
missing_pct = df[feature_cols].isnull().mean() * 100
high_missing = missing_pct[missing_pct > 50]

if len(high_missing) > 0:
    print(f"\n⚠️ Removing {len(high_missing)} features with >50% missing:")
    for col in high_missing.index:
        print(f"   - {col}: {high_missing[col]:.1f}% missing")
    feature_cols = [col for col in feature_cols if col not in high_missing.index]

print(f"\n✅ Final feature count: {len(feature_cols)}")
print(f"\nFeatures used:")
for i, col in enumerate(feature_cols, 1):
    print(f"   {i:2d}. {col}")

🔍 Selecting features...

✅ Available features: 48

⚠️ Removing 1 features with >50% missing:
   - circuit_std_position: 66.4% missing

✅ Final feature count: 47

Features used:
    1. max_throttle_ratio
    2. brake_max_g
    3. brake_avg_g
    4. avg_rainfall
    5. avg_track_temp
    6. avg_air_temp
    7. circuit_avg_position
    8. circuit_best_position
    9. circuit_worst_position
   10. circuit_sessions
   11. circuit_avg_race
   12. circuit_best_race
   13. recent_avg_position
   14. recent_best_position
   15. recent_worst_position
   16. form_trend
   17. races_in_window
   18. wet_avg_position
   19. dry_avg_position
   20. wet_sessions
   21. dry_sessions
   22. wet_dry_delta
   23. team_circuit_avg_position
   24. team_circuit_best_position
   25. team_circuit_sessions
   26. team_momentum
   27. team_recent_avg
   28. circuit_avg_position_change
   29. circuit_std_position_change
   30. circuit_abs_position_change
   31. circuit_max_gain
   32. circuit_max_loss
   33. cir

In [7]:
# Visualize missing data
missing_data = df[feature_cols].isnull().mean() * 100
missing_data = missing_data[missing_data > 0].sort_values(ascending=True).tail(15)

if len(missing_data) > 0:
    px.bar(
        missing_data,
        x=missing_data.values,
        y=missing_data.index,
        orientation='h',
        title='Missing Data by Feature (Top 15)',
        labels={'x': 'Missing %', 'y': 'Feature'}
    ).show()

## 4. Train/Test Split (Temporal)

In [ ]:
print("📊 Creating train/test split...\n")

# Temporal split: 2022-2023-2024 train, 2025 test
train_df = df[df['year'] <= 2024].copy()
test_df = df[df['year'] == 2025].copy()

print(f"✅ Train: {len(train_df)} rows (2022-2023)")
print(f"✅ Test:  {len(test_df)} rows (2024-2025)")

# Sprint weekend breakdown
train_sprint = train_df['is_sprint_weekend'].sum()
test_sprint = test_df['is_sprint_weekend'].sum()
print(f"\n🏃 Sprint weekends:")
print(f"   Train: {train_sprint} ({100*train_sprint/len(train_df):.1f}%)")
print(f"   Test:  {test_sprint} ({100*test_sprint/len(test_df):.1f}%)")

📊 Creating train/test split...

✅ Train: 1359 rows (2022-2023)
✅ Test:  419 rows (2024-2025)

🏃 Sprint weekends:
   Train: 0 (0.0%)
   Test:  0 (0.0%)


In [9]:
# Handle missing data
print("\n🔧 Handling missing data...")

imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(train_df[feature_cols])
X_test = imputer.transform(test_df[feature_cols])

print(f"✅ Features imputed: {X_train.shape}")
print(f"   Train: {X_train.shape}")
print(f"   Test:  {X_test.shape}")


🔧 Handling missing data...
✅ Features imputed: (1359, 47)
   Train: (1359, 47)
   Test:  (419, 47)


## 5. Model Training - Binary Classification

### 5.1 Helper Function

In [10]:
def train_binary_classifier(X_train, y_train, X_test, y_test, target_name):
    """
    Train and evaluate binary classifier.
    
    Returns:
        dict with model, metrics, predictions, importance
    """
    print(f"\n{'='*80}")
    print(f"🎯 TARGET: {target_name}")
    print(f"{'='*80}")
    
    # Class distribution
    print(f"\n📊 Class distribution:")
    print(f"   Train: {y_train.mean():.1%} positive ({y_train.sum()}/{len(y_train)})")
    print(f"   Test:  {y_test.mean():.1%} positive ({y_test.sum()}/{len(y_test)})")
    
    # Train model
    print(f"\n🤖 Training Random Forest...")
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(X_train, y_train)
    
    # Predictions
    y_pred_test = model.predict(X_test)
    y_pred_proba_test = model.predict_proba(X_test)[:, 1]
    
    # Metrics
    acc = accuracy_score(y_test, y_pred_test)
    prec = precision_score(y_test, y_pred_test, zero_division=0)
    rec = recall_score(y_test, y_pred_test, zero_division=0)
    f1 = f1_score(y_test, y_pred_test, zero_division=0)
    auc = roc_auc_score(y_test, y_pred_proba_test)
    
    print(f"\n📊 Test Performance:")
    print(f"   Accuracy:  {acc:.3f}")
    print(f"   Precision: {prec:.3f}")
    print(f"   Recall:    {rec:.3f}")
    print(f"   F1 Score:  {f1:.3f}")
    print(f"   ROC AUC:   {auc:.3f}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred_test)
    print(f"\n  Confusion Matrix:")
    print(f"    TN: {cm[0,0]:4d}  |  FP: {cm[0,1]:4d}")
    print(f"    FN: {cm[1,0]:4d}  |  TP: {cm[1,1]:4d}")
    
    # Feature importance
    importance_df = pd.DataFrame({
        'feature': feature_cols,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\n  Top 10 Features:")
    for idx, row in importance_df.head(10).iterrows():
        print(f"    {row['feature']:30s}: {row['importance']:.4f}")
    
    return {
        'model': model,
        'importance': importance_df,
        'metrics': {
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'auc': auc
        },
        'predictions': y_pred_test,
        'probabilities': y_pred_proba_test,
        'confusion_matrix': cm
    }

### 5.2 Q3 Qualification Model

In [11]:
# Train Q3 model
y_train_q3 = train_df['made_q3'].values
y_test_q3 = test_df['made_q3'].values

results_q3 = train_binary_classifier(
    X_train, y_train_q3, X_test, y_test_q3, 
    "Q3 Qualification (Top 10)"
)


🎯 TARGET: Q3 Qualification (Top 10)

📊 Class distribution:
   Train: 50.1% positive (681/1359)
   Test:  50.1% positive (210/419)

🤖 Training Random Forest...

📊 Test Performance:
   Accuracy:  0.783
   Precision: 0.796
   Recall:    0.762
   F1 Score:  0.779
   ROC AUC:   0.871

  Confusion Matrix:
    TN:  168  |  FP:   41
    FN:   50  |  TP:  160

  Top 10 Features:
    dry_avg_position              : 0.2270
    wet_avg_position              : 0.1606
    team_recent_avg               : 0.1529
    recent_avg_position           : 0.0411
    team_momentum                 : 0.0326
    driver_defensive_success_rate : 0.0297
    recent_worst_position         : 0.0215
    recent_best_position          : 0.0202
    wet_dry_delta                 : 0.0177
    circuit_best_position         : 0.0157


### 5.3 Top 3 Model

In [12]:
# Train Top 3 model
y_train_top3 = train_df['top3_quali'].values
y_test_top3 = test_df['top3_quali'].values

results_top3 = train_binary_classifier(
    X_train, y_train_top3, X_test, y_test_top3,
    "Top 3 in Qualifying (Podium)"
)


🎯 TARGET: Top 3 in Qualifying (Podium)

📊 Class distribution:
   Train: 15.0% positive (204/1359)
   Test:  15.0% positive (63/419)

🤖 Training Random Forest...

📊 Test Performance:
   Accuracy:  0.909
   Precision: 0.705
   Recall:    0.683
   F1 Score:  0.694
   ROC AUC:   0.941

  Confusion Matrix:
    TN:  338  |  FP:   18
    FN:   20  |  TP:   43

  Top 10 Features:
    dry_avg_position              : 0.2255
    wet_avg_position              : 0.1500
    team_recent_avg               : 0.1475
    recent_avg_position           : 0.0390
    driver_defensive_success_rate : 0.0316
    wet_dry_delta                 : 0.0243
    recent_worst_position         : 0.0238
    circuit_best_position         : 0.0230
    circuit_avg_position          : 0.0218
    circuit_worst_position        : 0.0209


## 6. Multi-Class Classification - Qualifying Round

In [13]:
print(f"\n{'='*80}")
print(f"🎯 TARGET: Qualifying Round (Multi-Class: Q1/Q2/Q3)")
print(f"{'='*80}")

y_train_q2 = train_df['quali_round_encoded'].values
y_test_q2 = test_df['quali_round_encoded'].values

# Class distribution
print(f"\n📊 Class distribution:")
for round_name, round_code in quali_round_map.items():
    train_pct = (y_train_q2 == round_code).mean()
    test_pct = (y_test_q2 == round_code).mean()
    print(f"   {round_name}: Train {train_pct:.1%}, Test {test_pct:.1%}")

# Train model
print(f"\n🤖 Training Random Forest (Multi-Class)...")
model_q2 = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=20,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

model_q2.fit(X_train, y_train_q2)

# Predictions
y_pred_q2 = model_q2.predict(X_test)

# Metrics
acc_q2 = accuracy_score(y_test_q2, y_pred_q2)
print(f"\n📊 Test Performance:")
print(f"   Overall Accuracy: {acc_q2:.3f}")

# Per-class metrics
print(f"\n  Per-Class Performance:")
for round_name, round_code in quali_round_map.items():
    mask = y_test_q2 == round_code
    if mask.sum() > 0:
        class_acc = (y_pred_q2[mask] == round_code).mean()
        print(f"    {round_name}: {class_acc:.3f} accuracy ({mask.sum()} samples)")

# Confusion matrix
cm_q2 = confusion_matrix(y_test_q2, y_pred_q2)
print(f"\n  Confusion Matrix:")
print(f"    Predicted:")
print(f"            Q1   Q2    Q3")
print(f"    Q1:   {cm_q2[0,0]:4d} {cm_q2[0,1]:4d} {cm_q2[0,2]:4d}")
print(f"    Q2:   {cm_q2[1,0]:4d} {cm_q2[1,1]:4d} {cm_q2[1,2]:4d}")
print(f"    Q3:   {cm_q2[2,0]:4d} {cm_q2[2,1]:4d} {cm_q2[2,2]:4d}")

# Feature importance
importance_q2 = pd.DataFrame({
    'feature': feature_cols,
    'importance': model_q2.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n  Top 10 Features:")
for idx, row in importance_q2.head(10).iterrows():
    print(f"    {row['feature']:30s}: {row['importance']:.4f}")

results_q2 = {
    'model': model_q2,
    'importance': importance_q2,
    'metrics': {'accuracy': acc_q2},
    'predictions': y_pred_q2,
    'confusion_matrix': cm_q2
}


🎯 TARGET: Qualifying Round (Multi-Class: Q1/Q2/Q3)

📊 Class distribution:
   Q1: Train 24.6%, Test 24.8%
   Q2: Train 25.3%, Test 25.1%
   Q3: Train 50.1%, Test 50.1%

🤖 Training Random Forest (Multi-Class)...

📊 Test Performance:
   Overall Accuracy: 0.594

  Per-Class Performance:
    Q1: 0.606 accuracy (104 samples)
    Q2: 0.143 accuracy (105 samples)
    Q3: 0.814 accuracy (210 samples)

  Confusion Matrix:
    Predicted:
            Q1   Q2    Q3
    Q1:     63   23   18
    Q2:     44   15   46
    Q3:     17   22  171

  Top 10 Features:
    dry_avg_position              : 0.2093
    wet_avg_position              : 0.1420
    team_recent_avg               : 0.1390
    recent_avg_position           : 0.0389
    team_momentum                 : 0.0354
    driver_defensive_success_rate : 0.0265
    wet_dry_delta                 : 0.0218
    brake_avg_g                   : 0.0217
    recent_best_position          : 0.0186
    brake_max_g                   : 0.0171


## 7. Visualizations

In [14]:
# Create outputs directory
Path('outputs/figures').mkdir(parents=True, exist_ok=True)
print("📊 Generating visualizations...\n")

📊 Generating visualizations...



### 7.1 Model Performance Comparison

In [15]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots


models = ['Q3 Qualification', 'Top 3 Quali', 'Q2 Multi-Class']
accuracies = [
    results_q3['metrics']['accuracy'],
    results_top3['metrics']['accuracy'],
    results_q2['metrics']['accuracy']
]
baselines = [0.50, 0.15, 0.333]
improvements = [a - b for a, b in zip(accuracies, baselines)]

# Top 5 Q3 features
top5_q3 = results_q3['importance'].head(5)

# Subplots layout
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "Model vs Baseline Accuracy",
        "Improvement Over Baseline",
        "Top 5 Features (Q3 Model)"
    ]
)

# -------------------------------
# 1. Accuracy comparison
# -------------------------------
x = np.arange(len(models))

fig.add_trace(
    go.Bar(
        x=models,
        y=baselines,
        name="Baseline",
        marker_color="lightcoral"
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=models,
        y=accuracies,
        name="Our Model",
        marker_color="skyblue"
    ),
    row=1, col=1
)

# Add annotations
for model, b, a in zip(models, baselines, accuracies):
    fig.add_annotation(
        x=model,
        y=b,
        text=f"{b:.1%}",
        yshift=12,
        showarrow=False,
        row=1, col=1
    )
    fig.add_annotation(
        x=model,
        y=a,
        text=f"{a:.1%}",
        yshift=12,
        showarrow=False,
        row=1, col=1
    )

fig.update_yaxes(range=[0, 1], row=1, col=1, title="Accuracy")

# -------------------------------
# 2. Improvement
# -------------------------------
fig.add_trace(
    go.Bar(
        x=models,
        y=improvements,
        marker_color="mediumseagreen"
    ),
    row=1, col=2
)

for model, imp in zip(models, improvements):
    fig.add_annotation(
        x=model,
        y=imp,
        text=f"+{imp:.1%}",
        yshift=12,
        showarrow=False,
        row=1, col=2
    )

fig.update_yaxes(title="Improvement (percentage points)", row=1, col=2)

# -------------------------------
# 3. Feature importance (horizontal)
# -------------------------------
fig.add_trace(
    go.Bar(
        x=top5_q3['importance'],
        y=top5_q3['feature'],
        orientation='h',
        marker_color="steelblue"
    ),
    row=1, col=3
)

fig.update_yaxes(autorange="reversed", row=1, col=3)
fig.update_xaxes(title="Importance", row=1, col=3)

# -------------------------------
# Layout
# -------------------------------
fig.update_layout(
    height=500,
    width=1500,
    barmode='group',
    showlegend=True,
)

fig.show()


### 7.2 Confusion Matrices

In [16]:
# Extract data
cm_q3 = results_q3['confusion_matrix']
cm_top3 = results_top3['confusion_matrix']
cm_q2 = results_q2['confusion_matrix']

# Labels
labels_q3 = ['Q2', 'Q3']
labels_top3 = ['P4+', 'Top3']
labels_q2 = ['Q1', 'Q2', 'Q3']

# Subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        f"Q3 Qualification<br>Accuracy: {results_q3['metrics']['accuracy']:.1%}",
        f"Top 3 Qualification<br>Accuracy: {results_top3['metrics']['accuracy']:.1%}",
        f"Qualifying Round<br>Accuracy: {results_q2['metrics']['accuracy']:.1%}"
    ]
)

# ---------------------------
# 1. Q3 heatmap
# ---------------------------
fig.add_trace(
    go.Heatmap(
        z=cm_q3,
        x=labels_q3,
        y=labels_q3,
        colorscale='Blues',
        showscale=False,
        text=cm_q3,
        texttemplate="%{text}"
    ),
    row=1, col=1
)

# ---------------------------
# 2. Top 3 heatmap
# ---------------------------
fig.add_trace(
    go.Heatmap(
        z=cm_top3,
        x=labels_top3,
        y=labels_top3,
        colorscale='Greens',
        showscale=False,
        text=cm_top3,
        texttemplate="%{text}"
    ),
    row=1, col=2
)

# ---------------------------
# 3. Q1/Q2/Q3 multi-class heatmap
# ---------------------------
fig.add_trace(
    go.Heatmap(
        z=cm_q2,
        x=labels_q2,
        y=labels_q2,
        colorscale='Oranges',
        showscale=False,
        text=cm_q2,
        texttemplate="%{text}"
    ),
    row=1, col=3
)

# ---------------------------
# Layout
# ---------------------------
fig.update_xaxes(title="Predicted")
fig.update_yaxes(title="Actual")

fig.update_layout(
    height=450,
    width=1500,
    margin=dict(t=100),
)

fig.show()


### 7.3 Feature Importance Comparison

In [17]:
import plotly.graph_objs as go
from plotly.subplots import make_subplots

# Extract top 10 features
top10_q3 = results_q3['importance'].head(10)
top10_top3 = results_top3['importance'].head(10)

# Create subplot layout
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Top 10 Features - Q3 Model",
        "Top 10 Features - Top 3 Model"
    ]
)

# -----------------------------------------
# 1. Q3 model feature importance
# -----------------------------------------
fig.add_trace(
    go.Bar(
        x=top10_q3['importance'],
        y=top10_q3['feature'],
        orientation='h',
        marker_color='steelblue'
    ),
    row=1, col=1
)

fig.update_yaxes(autorange="reversed", row=1, col=1)
fig.update_xaxes(title="Importance", row=1, col=1)

# -----------------------------------------
# 2. Top 3 model feature importance
# -----------------------------------------
fig.add_trace(
    go.Bar(
        x=top10_top3['importance'],
        y=top10_top3['feature'],
        orientation='h',
        marker_color='darkgreen'
    ),
    row=1, col=2
)

fig.update_yaxes(autorange="reversed", row=1, col=2)
fig.update_xaxes(title="Importance", row=1, col=2)

# -----------------------------------------
# Layout
# -----------------------------------------
fig.update_layout(
    height=500,
    width=1500,
    showlegend=False,
    margin=dict(t=80)
)

fig.show()


### 7.4 ROC Curves

In [18]:
import plotly.graph_objs as go

# Compute ROC curves
fpr_q3, tpr_q3, _ = roc_curve(y_test_q3, results_q3['probabilities'])
fpr_top3, tpr_top3, _ = roc_curve(y_test_top3, results_top3['probabilities'])

auc_q3 = results_q3['metrics']['auc']
auc_top3 = results_top3['metrics']['auc']

# Create figure
fig = go.Figure()

# -----------------------------
# Q3 ROC
# -----------------------------
fig.add_trace(
    go.Scatter(
        x=fpr_q3,
        y=tpr_q3,
        mode='lines',
        line=dict(width=2, color='steelblue'),
        name=f"Q3 Qualification (AUC = {auc_q3:.3f})"
    )
)

# -----------------------------
# Top 3 ROC
# -----------------------------
fig.add_trace(
    go.Scatter(
        x=fpr_top3,
        y=tpr_top3,
        mode='lines',
        line=dict(width=2, color='darkgreen'),
        name=f"Top 3 Finish (AUC = {auc_top3:.3f})"
    )
)

# -----------------------------
# Random baseline
# -----------------------------
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode='lines',
        line=dict(width=1, dash='dash', color='black'),
        name="Random Classifier"
    )
)

# -----------------------------
# Layout
# -----------------------------
fig.update_layout(
    title="ROC Curves - Classification Models",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    width=900,
    height=700,
    template="simple_white",
)

# Add grid (Plotly hides by default)
fig.update_xaxes(showgrid=True, gridcolor='lightgrey')
fig.update_yaxes(showgrid=True, gridcolor='lightgrey')

fig.show()


## 8. Sprint Weekend Analysis

In [19]:
if test_sprint > 0:
    print(f"\n{'='*80}")
    print(f"🏃 SPRINT WEEKEND ANALYSIS")
    print(f"{'='*80}")
    
    sprint_mask = test_df['is_sprint_weekend'].values
    normal_mask = ~sprint_mask
    
    print(f"\n📊 Q3 Classification:")
    if sprint_mask.sum() > 0:
        acc_sprint = accuracy_score(y_test_q3[sprint_mask], results_q3['predictions'][sprint_mask])
        acc_normal = accuracy_score(y_test_q3[normal_mask], results_q3['predictions'][normal_mask])
        
        print(f"   Sprint weekends: {acc_sprint:.3f} ({sprint_mask.sum()} races)")
        print(f"   Normal weekends: {acc_normal:.3f} ({normal_mask.sum()} races)")
        print(f"   Difference: {acc_normal - acc_sprint:+.3f}")
else:
    print("\n⚠️ No sprint weekend data detected")


⚠️ No sprint weekend data detected


## 9. Save Models

In [ ]:
import os
base_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

print(f"\n{'='*80}")
print(f"💾 SAVING MODELS")
print(f"{'='*80}\n")

model_dir = os.path.join(base_dir, 'models/legacy')
os.makedirs(model_dir, exist_ok=True)

# Save models
joblib.dump(results_top3['model'], os.path.join(model_dir, 'top3_classifier.pkl'))
joblib.dump(results_q3['model'], os.path.join(model_dir, 'q3_classifier.pkl'))
joblib.dump(results_q2['model'], os.path.join(model_dir, 'q2_classifier.pkl'))
print("✅ Models saved to models/")

# Save feature importance
results_top3['importance'].to_csv(os.path.join(model_dir, 'top3_classifier.csv'), index=False)
results_q3['importance'].to_csv(os.path.join(model_dir, 'q3_classifier.csv'), index=False)
results_q2['importance'].to_csv(os.path.join(model_dir, 'q2_classifier.csv'), index=False)
print("✅ Feature importance saved")

# Save metadata
metadata = {
    'models': {
        'q3_binary': {
            'accuracy': float(results_q3['metrics']['accuracy']),
            'precision': float(results_q3['metrics']['precision']),
            'recall': float(results_q3['metrics']['recall']),
            'auc': float(results_q3['metrics']['auc'])
        },
        'top3_binary': {
            'accuracy': float(results_top3['metrics']['accuracy']),
            'precision': float(results_top3['metrics']['precision']),
            'recall': float(results_top3['metrics']['recall']),
            'auc': float(results_top3['metrics']['auc'])
        },
        'q2_multiclass': {
            'accuracy': float(results_q2['metrics']['accuracy'])
        }
    },
    'features': feature_cols,
    'train_size': len(train_df),
    'test_size': len(test_df),
    'sprint_weekends_test': int(test_sprint)
}

with open(os.path.join(model_dir,'classification_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)
print("✅ Metadata saved")


💾 SAVING MODELS

✅ Models saved to models/
✅ Feature importance saved
✅ Metadata saved


## 10. Final Summary

In [21]:
print(f"\n{'='*80}")
print(f"📊 FINAL SUMMARY")
print(f"{'='*80}\n")

print(f"🎯 Model Performance (Test Set):")
print(f"   Top 3 Classification: {results_top3['metrics']['accuracy']:.1%} accuracy")
print(f"   Q3 Classification:    {results_q3['metrics']['accuracy']:.1%} accuracy")
print(f"   Q2 Multi-Class:       {results_q2['metrics']['accuracy']:.1%} accuracy")

print(f"\n✅ Baselines Beaten:")
print(f"   Top3: 15.0% → {results_top3['metrics']['accuracy']:.1%} (+{100*(results_top3['metrics']['accuracy']-0.15):.1f} points)")
print(f"   Q3:   50.0% → {results_q3['metrics']['accuracy']:.1%} (+{100*(results_q3['metrics']['accuracy']-0.50):.1f} points)")
print(f"   Q2:   33.3% → {results_q2['metrics']['accuracy']:.1%} (+{100*(results_q2['metrics']['accuracy']-0.333):.1f} points)")

print(f"\n🏆 Ready Models:")
print(f"   ✅ Top 3 Classification (binary)")
print(f"   ✅ Q3 Classification (binary)")
print(f"   ✅ Qualifying Round (multi-class)")

print(f"\n📁 Files Generated:")
print(f"   - models/top3_classifier.pkl")
print(f"   - models/q3_classifier.pkl")
print(f"   - models/q2_classifier.pkl")
print(f"   - models/feature_importance_*.csv")
print(f"   - models/classification_metadata.json")

print(f"\n{'='*80}")
print(f"✅ CLASSIFICATION COMPLETE - READY!")
print(f"{'='*80}")


📊 FINAL SUMMARY

🎯 Model Performance (Test Set):
   Top 3 Classification: 90.9% accuracy
   Q3 Classification:    78.3% accuracy
   Q2 Multi-Class:       59.4% accuracy

✅ Baselines Beaten:
   Top3: 15.0% → 90.9% (+75.9 points)
   Q3:   50.0% → 78.3% (+28.3 points)
   Q2:   33.3% → 59.4% (+26.1 points)

🏆 Ready Models:
   ✅ Top 3 Classification (binary)
   ✅ Q3 Classification (binary)
   ✅ Qualifying Round (multi-class)

📁 Files Generated:
   - models/top3_classifier.pkl
   - models/q3_classifier.pkl
   - models/q2_classifier.pkl
   - models/feature_importance_*.csv
   - models/classification_metadata.json

✅ CLASSIFICATION COMPLETE - READY!
